# 🌱 KOOBO — Entraînement du modèle de détection HORS-LIGNE

Ce notebook entraîne un **modèle léger** (MobileNetV3-Small) de détection de maladies des plantes,
puis l'exporte au format **TensorFlow.js** pour tourner **dans le navigateur** (hors-ligne, sans API).

## ✅ À faire AVANT de lancer
1. Menu **Exécution → Modifier le type d'exécution → Accélérateur matériel : GPU** (T4).
2. Puis **Exécution → Tout exécuter**.

Aucun téléchargement manuel : le jeu de données **PlantVillage** est récupéré automatiquement.
À la fin, un fichier **`koobo_web_model.zip`** se télécharge sur votre ordinateur.


## 1. Installation


In [ ]:
# tensorflow & tensorflow_datasets sont déjà présents dans Colab.
# On installe le convertisseur TensorFlow.js.
!pip -q install tensorflowjs


## 2. Imports & configuration


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import json, os, shutil
print('TensorFlow', tf.__version__)
print('GPU dispo :', tf.config.list_physical_devices('GPU') or 'AUCUN (activez le GPU !)')

IMG = 224           # taille d'entrée du modèle
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE


## 3. Jeux de données (3 sources, téléchargées automatiquement)
On combine **trois** datasets ouverts via TensorFlow Datasets (aucun téléchargement manuel) :
- **PlantVillage** (~54 000 images, 38 classes) — référence multi-cultures
- **Cassava** (~9 400 images, 5 classes) — manioc, culture clé en Afrique
- **Beans** (~1 300 images, 3 classes) — haricot, images terrain (Ouganda)

Chaque dataset garde ses propres classes (préfixées par sa source). Split 80 % / 10 % / 10 % chacun, puis fusion.

> ⏳ Le téléchargement (~2–3 Go au total) peut prendre quelques minutes la 1re fois.
> 💡 Essai rapide : remplacez `train[:80%]` par `train[:10%]` (et adaptez val/test).


In [ ]:
DATASETS = ['plant_village', 'cassava', 'beans']   # tous via TFDS (sans Kaggle)

CLASSES = []
train_parts, val_parts, test_parts = [], [], []

for name in DATASETS:
    (d_tr, d_va, d_te), info = tfds.load(
        name, split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
        as_supervised=True, with_info=True)
    local_names = info.features['label'].names
    offset = len(CLASSES)                       # décalage pour des indices globaux uniques
    CLASSES += [f'{name}:{n}' for n in local_names]
    o = offset
    train_parts.append(d_tr.map(lambda x, y, o=o: (x, y + o), AUTOTUNE))
    val_parts.append(d_va.map(lambda x, y, o=o: (x, y + o), AUTOTUNE))
    test_parts.append(d_te.map(lambda x, y, o=o: (x, y + o), AUTOTUNE))
    print(f'{name}: {len(local_names)} classes')

NUM = len(CLASSES)
print('TOTAL :', NUM, 'classes')

def concat(parts):
    out = parts[0]
    for p in parts[1:]:
        out = out.concatenate(p)
    return out

raw_train, raw_val, raw_test = concat(train_parts), concat(val_parts), concat(test_parts)


## 4. Pipeline de données + augmentation
L'augmentation est appliquée **dans le pipeline** (pas dans le modèle) → modèle propre, 100 % convertible en TF.js.
On garde les images en **[0, 255]** : MobileNetV3 (`include_preprocessing=True`) fait la normalisation lui-même.


In [ ]:
def prep(img, label):
    img = tf.image.resize(img, (IMG, IMG))
    img = tf.cast(img, tf.float32)   # reste en 0-255
    return img, label

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
])

train = (raw_train.map(prep, AUTOTUNE).shuffle(3000).batch(BATCH)
         .map(lambda x, y: (augment(x, training=True), y), AUTOTUNE).prefetch(AUTOTUNE))
val   = raw_val.map(prep, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
test  = raw_test.map(prep, AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)


## 5. Modèle (transfer learning — MobileNetV3-Small)


In [ ]:
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet',
    include_preprocessing=True)   # le modèle normalise lui-même les entrées 0-255
base.trainable = False

inputs = tf.keras.Input((IMG, IMG, 3))           # attend des pixels 0-255
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 6. Entraînement — Phase 1 (base gelée)


In [ ]:
model.fit(train, validation_data=val, epochs=8)


## 7. Entraînement — Phase 2 (fine-tuning, lr faible)


In [ ]:
base.trainable = True
for layer in base.layers[:-40]:   # on n'affine que les dernières couches
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train, validation_data=val, epochs=4)


## 8. Évaluation


In [ ]:
loss, acc = model.evaluate(test)
print('Précision sur le test :', round(acc * 100, 2), '%')


## 9. Sauvegarde du modèle + des classes


In [ ]:
model.save('koobo_disease.keras')
with open('classes.json', 'w', encoding='utf-8') as f:
    json.dump(CLASSES, f, ensure_ascii=False)
print('Modèle Keras et classes.json sauvegardés.')


## 10. Conversion en TensorFlow.js (pour le navigateur)
Produit un dossier `web_model/` : `model.json` + fichiers `.bin` + `classes.json`.


In [ ]:
import tensorflowjs as tfjs
shutil.rmtree('web_model', ignore_errors=True)
tfjs.converters.save_keras_model(model, 'web_model')
shutil.copy('classes.json', 'web_model/classes.json')
print('Fichiers générés :', os.listdir('web_model'))


## 11. Télécharger le modèle web
Récupère `koobo_web_model.zip`. Décompressez-le dans **`web/public/model/`** de votre projet.


In [ ]:
shutil.make_archive('koobo_web_model', 'zip', 'web_model')
from google.colab import files
files.download('koobo_web_model.zip')


## 12. (Optionnel) Test rapide d'une prédiction


In [ ]:
import numpy as np
for imgs, labels in test.take(1):
    preds = model.predict(imgs)
    for i in range(3):
        print('Vrai :', CLASSES[labels[i].numpy()], '| Prédit :', CLASSES[int(np.argmax(preds[i]))],
              '(', round(float(np.max(preds[i])) * 100, 1), '%)')


---
### ➡️ Et après ?
1. Décompressez `koobo_web_model.zip` → mettez son contenu dans **`web/public/model/`**
   (vous devez avoir `web/public/model/model.json`, les `.bin` et `classes.json`).
2. Reconstruisez le frontend : `npm run build`.
3. La page **Détection → onglet « Hors-ligne »** chargera ce modèle et fera le diagnostic
   directement dans le navigateur, sans internet ni quota IA.
4. Re-téléversez `web/dist` sur cPanel. Terminé !
